# cathtriage - NB03 only: acquisition (TabPFN) + offline analysis. ~2.5-3.5 h
GPU ON, Internet ON. Secrets: `TABPFN_TOKEN` (+ `GITHUB_TOKEN` if private repo). Edit `GITHUB_URL` in cell 2.


## [CELL 1] install dependencies

In [ ]:
import os, sys, subprocess
if not os.environ.get("SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn", "catboost", "shap",
                    "openpyxl", "pytest", "scikit-learn", "matplotlib"], check=False)
print("deps ok")


## [CELL 2] clone / update repo, paths, secrets

In [ ]:
import os, sys, subprocess, json, time, warnings, contextlib, traceback
warnings.filterwarnings("ignore")
GITHUB_URL = os.environ.get("GITHUB_URL", "https://github.com/<your-username>/cathtriage.git")   # <-- EDIT
REPO    = os.environ.get("REPO", "/kaggle/working/cathtriage")
SRC, RESULTS, DATA_DIR = f"{REPO}/src", f"{REPO}/results", f"{REPO}/data"
QUICK   = os.environ.get("QUICK", "0") == "1"       # QUICK=1 -> 1 repeat / 3 folds smoke mode
REPS    = 1 if QUICK else 5

gh_token = None
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    try: os.environ["TABPFN_TOKEN"] = sec.get_secret("TABPFN_TOKEN")
    except Exception: print("no TABPFN_TOKEN secret found")
    try: gh_token = sec.get_secret("GITHUB_TOKEN")
    except Exception: pass
except Exception:
    pass

if not os.environ.get("SKIP_CLONE"):
    url = GITHUB_URL.replace("https://", f"https://{gh_token}@") if gh_token else GITHUB_URL
    if os.path.isdir(f"{REPO}/.git"):
        r = subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], capture_output=True, text=True)
    else:
        r = subprocess.run(["git", "clone", "--depth", "1", url, REPO], capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(gh_token or "@@@", "***")[-400:])
sys.path.insert(0, SRC); os.makedirs(RESULTS, exist_ok=True)

@contextlib.contextmanager
def stage(name):
    t0 = time.time(); print(f"\n{'=' * 12} {name} {'=' * 12}")
    try: yield
    except Exception:
        print(f"!!!!!! {name} FAILED !!!!!!"); traceback.print_exc()
    finally: print(f"[{name}] {(time.time() - t0) / 60:.1f} min")


## [CELL 3] copy dataset, load, engineer features, check TabPFN, write run metadata

In [ ]:
import shutil, pickle
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
DATA_SRC = os.environ.get("DATA_SRC", "/kaggle/input/datasets/nikreddyvempalli/dataset/extention of Z-Alizadeh sani dataset.xlsx")
os.makedirs(DATA_DIR, exist_ok=True); DATA = f"{DATA_DIR}/zalizadeh_sani_ext.xlsx"   # data/ is gitignored: never redistribute
shutil.copy(DATA_SRC, DATA)

from data import load
from features import engineer
from baselines import benchmark, report, compare      # shared by NB01/NB02/NB04
from sklearn.metrics import roc_auc_score
X, y8, Yv, names, gcols, costs, gnames, free = load(DATA)
X2, names2, gcols2, added = engineer(X, names, gcols, gnames, free)
print("raw:", X.shape, "| engineered:", X2.shape, "| added:", added)

try:
    import torch; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
TABPFN_OK = False
try:
    from tabpfn import TabPFNClassifier
    TabPFNClassifier(device=DEVICE, n_estimators=2).fit(X[:80, :6], Yv[:80, 0]).predict_proba(X[:5, :6])
    TABPFN_OK = True
except Exception as e:
    print("TabPFN UNAVAILABLE ->", type(e).__name__, str(e)[:300],
          "\n   Fix: log in at https://ux.priorlabs.ai, accept the licence, copy your API key, add it as Kaggle secret TABPFN_TOKEN."
          "\n   Falling back to the logreg backend for acquisition (results will NOT be the headline numbers).")
print("DEVICE:", DEVICE, "| TABPFN_OK:", TABPFN_OK)

import platform, sklearn
meta = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "python": platform.python_version(), "sklearn": sklearn.__version__,
        "device": DEVICE, "tabpfn_ok": TABPFN_OK, "quick": QUICK, "n": int(len(X)), "reps": REPS,
        "git": subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()}
for pkg in ("catboost", "tabpfn", "torch"):
    try: meta[pkg] = __import__(pkg).__version__
    except Exception: pass
json.dump(meta, open(f"{RESULTS}/run_meta.json", "w"), indent=2); print(meta)

# ---- freshness guard: fail in seconds (not after a 2-hour stage) if GitHub is behind your local files ----
import inspect, importlib
import evaluate as _ev, baselines as _bl, posterior as _po
_problems = []
if "save_traj" not in inspect.signature(_ev.run).parameters: _problems.append("src/evaluate.py (save_traj)")
if "_sig" not in inspect.getsource(_bl.make): _problems.append("src/baselines.py (*_sig models)")
if "max_cache" not in inspect.signature(_po.SubsetPosterior.__init__).parameters: _problems.append("src/posterior.py (max_cache)")
try: importlib.import_module("analysis")
except Exception: _problems.append("src/analysis.py (missing)")
if "objective" not in inspect.signature(importlib.import_module("acquire").trajectory).parameters: _problems.append("src/acquire.py (objective)")
assert not _problems, "STALE REPO - git add/commit/push these, then rerun: " + ", ".join(_problems)
print("code freshness OK")


## [NB00] sanity: tests, leak checks, label definition, RWMA eyeball

In [ ]:
from checks import assert_no_leak, permutation_check, leakage_canary
with stage("NB00 sanity"):
    print("joint-state counts:", np.bincount(y8, minlength=8), "| prevalence LAD/LCX/RCA:", Yv.mean(0).round(3))
    raw = pd.read_excel(DATA)
    cath_cad = raw["Cath"].astype(str).str.strip().str.lower().eq("cad").astype(int).values
    mism = np.where(cath_cad != Yv.max(1))[0]
    print(f"rows where Cath-CAD != any-vessel>=50%: {mism.tolist()}")
    if len(mism): print(raw.loc[mism, ["Age", "LAD", "LCX", "RCA", "Cath"]])
    print("\nRWMA region vs P(stenosis) per vessel  (EDA ONLY - never used to build features):")
    print(pd.DataFrame({v: pd.Series(Yv[:, i]).groupby(raw["Region RWMA"].values).mean() for i, v in enumerate(["LAD", "LCX", "RCA"])}).round(2))
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-x", "-p", "no:warnings", "tests"], cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-300:]); assert r.returncode == 0, "unit tests failed"
    assert_no_leak(names); assert_no_leak(names2)
    perm, canary = permutation_check(X, y8, Yv), leakage_canary(X, y8, Yv)
    print(f"permutation AUC (want ~0.50): {perm:.3f} | leak-canary CAD AUC (want ~1.0): {canary:.3f}")
    assert abs(perm - 0.5) < 0.07 and canary > 0.97, "harness sanity check failed - do NOT trust downstream numbers"


## [NB03] THE headline: adaptive acquisition (VOI) vs baselines + conformal sets

In [ ]:
from evaluate import run, summarise
BACKEND = "tabpfn" if TABPFN_OK else "logreg"
ACQ_DEV = DEVICE if BACKEND == "tabpfn" else "cpu"
with stage(f"NB03b acquisition ({BACKEND})"):
    TRAJ = f"{RESULTS}/nb03_traj_{BACKEND}.pkl"          # checkpointed after every fold
    df = run(X2, y8, Yv, gcols2, costs, free, backend=BACKEND, device=ACQ_DEV,
             policies=("voi", "voi_cad", "static", "random"), taus=(0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99),
             n_splits=3 if QUICK else 5, n_repeats=1, K=8, save_traj=TRAJ)
    df.to_csv(f"{RESULTS}/nb03_acq_{BACKEND}.csv", index=False)
    g = df.groupby(["policy", "tau"]).agg(cost=("cost", "mean"), auc=("cad_auc", "mean"), sd=("cad_auc", "std"),
                                          LAD_cov=("LAD_cov", "mean"), LCX_cov=("LCX_cov", "mean"), RCA_cov=("RCA_cov", "mean"),
                                          ambiguous=("ambiguous_rate", "mean"), singleton=("singleton_rate", "mean")).round(3).reset_index()
    g.to_csv(f"{RESULTS}/nb03_frontier_table.csv", index=False); print(g.to_string())
    plt.figure(figsize=(6, 4))
    for pol, d in g.groupby("policy"): plt.errorbar(d.cost, d.auc, d.sd, marker="o", capsize=2, label=pol)
    plt.xlabel("mean test cost per patient (relative units)"); plt.ylabel("CAD AUC"); plt.legend(); plt.title(f"Cost-accuracy frontier ({BACKEND})")
    plt.tight_layout(); plt.savefig(f"{RESULTS}/frontier_{BACKEND}.png", dpi=200); plt.close()
    target = 0.98 * g.auc.max()            # headline: cheapest operating point reaching 98% of best AUC
    for pol, d in g.groupby("policy"):
        ok = d[d.auc >= target]
        print(f"{pol:9s} cost to reach AUC>={target:.3f}: {ok.cost.min() if len(ok) else float('nan'):.2f}")


## [NB03c] offline re-analysis of saved trajectories (CPU, seconds): policy test, stopping rules, cross-conformal

In [ ]:
from analysis import stepwise, frontier, paired_policy_test
with stage("NB03c offline analysis"):
    trajs = pickle.load(open(TRAJ, "rb"))
    sw = stepwise(trajs, Yv); sw.to_csv(f"{RESULTS}/nb03c_stepwise.csv", index=False)
    print(sw.pivot(index="step", columns="policy", values="CAD_auc").to_string())
    print(sw.pivot(index="step", columns="policy", values="cost").round(2).to_string())
    plt.figure(figsize=(6, 4))
    for pol, d in sw.groupby("policy"): plt.plot(d.cost, d.CAD_auc, marker="o", label=pol)
    plt.xlabel("mean cumulative cost"); plt.ylabel("CAD AUC (pooled OOF)"); plt.legend(); plt.title("Ordering quality, stopping-rule free")
    plt.tight_layout(); plt.savefig(f"{RESULTS}/stepwise_{BACKEND}.png", dpi=200); plt.close()
    for a_, b_ in [("voi", "static"), ("voi_cad", "static"), ("voi_cad", "voi"), ("voi", "random")]:
        for st in (2, 3, 4, 5):
            d_, ci_, p_ = paired_policy_test(trajs, Yv, a_, b_, st)
            print(f"{a_:8s} - {b_:7s} step {st}: dAUC={d_:+.4f} CI=({ci_[0]:+.4f},{ci_[1]:+.4f}) p={p_:.3f}")
    fr = frontier(trajs, Yv); fr.to_csv(f"{RESULTS}/nb03c_frontier_conformal.csv", index=False)
    key = ["policy", "rule", "mode", "mondrian", "alpha"]
    show = fr[(fr.tau == 0.9)].groupby(key)[["cost", "CAD_auc", "LAD_cov", "LCX_cov", "RCA_cov", "singleton"]].mean().round(3)
    print(show.to_string())


## [DONE] zip results for easy download

In [ ]:
shutil.make_archive("/kaggle/working/results_bundle" if os.path.isdir("/kaggle/working") else f"{REPO}/results_bundle", "zip", RESULTS)
print("files:", sorted(os.listdir(RESULTS)))
